In [69]:
# use sentence-transformer for embedding model also used openai embedding model
# for the documents we will load a wikipedia page
# https://www.jcchouinard.com/wikipedia-api/



In [70]:
import os
from dotenv import load_dotenv
import openai
load_dotenv()
openai.api_key = os.getenv("Openai_API_KEY")
# client = os.environ['Openai_API_KEY']= os.getenv("Openai_API_KEY")


#### Fetch Text content using wikipedia

In [71]:
# Load Data
from wikipediaapi import Wikipedia
wiki = Wikipedia('MYApp/0.0', 'en')
data = wiki.page('Large_language_models').text
data1 = wiki.page('Large_language_models').title


In [72]:

# print(data)


##### Perform Chunking

In [73]:
paragraph = data.split('\n\n')
paragraph


['A large language model (LLM) is a type of machine learning model designed for natural language processing tasks such as language generation. As language models, LLMs acquire these abilities by learning statistical relationships from vast amounts of text during a self-supervised and semi-supervised training process.\nThe largest and most capable LLMs are artificial neural networks built with a decoder-only transformer-based architecture, enabling efficient processing and generation of large-scale text data. Modern models can be fine-tuned for specific tasks or guided by prompt engineering. These models acquire predictive power regarding syntax, semantics, and ontologies inherent in human language corpora, but they also inherit inaccuracies and biases present in the data they are trained in.',
 'History\nBefore 2017, there were a few language models that were large as compared to capacities then available. In the 1990s, the IBM alignment models pioneered statistical language modelling.

In [ ]:
import textwrap
for i,p in enumerate(paragraph):
    wraped_text = textwrap.fill(p,width=200)

    print("-----------------------------------------------")
    print(wraped_text)
    print("-----------------------------------------------")
    # print(f"Paragraph {i+1}:\n{wraped_text}\n")


-----------------------------------------------
A large language model (LLM) is a type of machine learning model designed for natural language processing tasks such as language generation. As language models, LLMs acquire these abilities by
learning statistical relationships from vast amounts of text during a self-supervised and semi-supervised training process. The largest and most capable LLMs are artificial neural networks built with
a decoder-only transformer-based architecture, enabling efficient processing and generation of large-scale text data. Modern models can be fine-tuned for specific tasks or guided by prompt
engineering. These models acquire predictive power regarding syntax, semantics, and ontologies inherent in human language corpora, but they also inherit inaccuracies and biases present in the data
they are trained in.
-----------------------------------------------
-----------------------------------------------
History Before 2017, there were a few language models th

##### Embed the Chunks

In [75]:
#load the Embedding Model
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("Alibaba-NLP/gte-base-en-v1.5",trust_remote_code=True)


##### Embedd the Document

In [76]:
# doc_embed = model.encode(paragraph,normalize_embeddings=True)
# import torch
# model.to(torch.device("cpu"))
doc_embed = model.encode(paragraph, normalize_embeddings=True)


In [77]:
doc_embed.shape


(51, 768)

In [78]:
# doc_embed[0]


##### embedd the query

In [79]:
query = "tell me about th elarge language model"
query_model = model.encode(query,normalize_embeddings=True)


In [80]:
query_model.shape


(768,)

##### find the closet paragraph of  the query

In [81]:
# doc_embed is basically a vector store

import numpy as np
similarity = np.dot(doc_embed,query_model.T)


In [82]:
similarity


array([0.6489255 , 0.5363411 , 0.5588774 , 0.34897813, 0.40211254,
       0.32979414, 0.3987603 , 0.28809628, 0.5248716 , 0.31988984,
       0.45963418, 0.40886563, 0.37971437, 0.42401406, 0.43398955,
       0.361334  , 0.37477458, 0.4204013 , 0.45605692, 0.34496993,
       0.49271291, 0.3120604 , 0.37977478, 0.2892632 , 0.40843397,
       0.4541643 , 0.40314165, 0.42553264, 0.3503499 , 0.30668342,
       0.38452193, 0.6217507 , 0.5211388 , 0.5381683 , 0.42540044,
       0.35875463, 0.55959964, 0.4124143 , 0.47096103, 0.45964497,
       0.25568286, 0.32334   , 0.5052105 , 0.357385  , 0.53460884,
       0.51726246, 0.46407753, 0.38292113, 0.43835586, 0.4514613 ,
       0.511502  ], dtype=float32)

In [83]:
similarity.shape


(51,)

In [ ]:
# top_k = np.argsort(similarity,axis=0)
# top_k


In [85]:
#rank them based on the  similarity score and choose top k
top_k = np.argsort(similarity,axis=0)[-3:][::-1].tolist()


In [86]:
top_k


[0, 31, 36]

In [87]:
#reverse indexing process
most_similar_documents= [paragraph[idx] for idx in top_k]


In [88]:
Contxt = ""
for i ,p in enumerate(most_similar_documents):
    wraped_text= textwrap.fill(p,width=200)

    print("============================")
    print(wraped_text)
    print("============================")
    Contxt += wraped_text+ "\n\n"


A large language model (LLM) is a type of machine learning model designed for natural language processing tasks such as language generation. As language models, LLMs acquire these abilities by
learning statistical relationships from vast amounts of text during a self-supervised and semi-supervised training process. The largest and most capable LLMs are artificial neural networks built with
a decoder-only transformer-based architecture, enabling efficient processing and generation of large-scale text data. Modern models can be fine-tuned for specific tasks or guided by prompt
engineering. These models acquire predictive power regarding syntax, semantics, and ontologies inherent in human language corpora, but they also inherit inaccuracies and biases present in the data
they are trained in.
reported arithmetics, decoding the International Phonetic Alphabet, unscrambling a word's letters, disambiguate word in context, converting spatial words, cardinal directions (for example, replying
"n

- we have query and relevant chunks now feed thoes into the llm and get the user reponse


In [89]:
querry = "tell me about LLM"


- the CONTEXT = Context is all of the top 3 similar chunks put them together and these are different paragraph and create the prompt

In [90]:
#prompt
prompt = f"""
use the following CONTEXT to answer the QUESTION at the end.
if you dont know the ans ,just say  you dont know,dont try to make up an answer.
CONTEXT :{Contxt}
QUESTION : {querry}
"""


In [97]:
#Response
response = openai.chat.completions.create(model= "gpt-4",messages=[{
    "role":"user","content":prompt}])


In [104]:
response


ChatCompletion(id='chatcmpl-Ai0CphBbtC5gRPNIsqKFXn09Vniwi', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A Large Language Model (LLM) is a machine learning model specifically created for tasks related to natural language processing, such as language generation. It acquires abilities required for these tasks through a self-supervised and semi-supervised training process, where it learns statistical relationships from a large amount of textual data. The largest and more capable LLMs are artificial neural networks that use a decoder-only transformer-based architecture, which makes processing and generating large-scale text data efficient. These models can be fine-tuned for specific tasks or managed through prompt engineering. They have predictive power regarding syntax, semantics, and ontologies present in human language corpora, but are also prone to absorbing inaccuracies and biases present in the training data. Modern LLMs can per

In [109]:
content = response.choices[0].message.content
print(content)


A Large Language Model (LLM) is a machine learning model specifically created for tasks related to natural language processing, such as language generation. It acquires abilities required for these tasks through a self-supervised and semi-supervised training process, where it learns statistical relationships from a large amount of textual data. The largest and more capable LLMs are artificial neural networks that use a decoder-only transformer-based architecture, which makes processing and generating large-scale text data efficient. These models can be fine-tuned for specific tasks or managed through prompt engineering. They have predictive power regarding syntax, semantics, and ontologies present in human language corpora, but are also prone to absorbing inaccuracies and biases present in the training data. Modern LLMs can perform a range of tasks, examples of which include decoding the International Phonetic Alphabet, unscrambling letters in a word, identifying offensive content, and